# Do we need the rotation model? High-pass check on three HI epochs

The package subtracts a DiskFit rotation model (flat disk, no warp, no radial
flow) before taking velocity gradients. Errors in that model leave smooth,
radius-dependent residual gradients of a few km/s/kpc, the same size as the
CO-bright divergence signal. This notebook repeats the divergence and shear
statistics with four large-scale subtractions:

- **rotation model**: moment-1 minus the model (the package default),
- **no subtraction**: moment-1 as observed,
- **high-pass 500 pc / 1000 pc**: moment-1 minus a Gaussian low-pass of itself
 (`flows.highpass_velocity_field`), which removes rotation, PA twist,
  axisymmetric radial flow and centre error without a model.

on three HI data sets: C+D+GBT (21.4"), 14B+17B (8.2") and A+B+C+D+GBT (5.3").
If the CO statistics agree across the four, the model does not matter. Where
they differ, the difference measures the large-scale systematic.

Each mode also reports the median divergence of **all** pixels in the CO
footprint, as a control: an offset shared by all gas is not a CO signal.

The high-pass is not valid within about one kernel width of the centre.

In [ ]:
import gc, os, pickle, sys, time, traceback
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from exgal_flows import FlowAnalysis

DATA = "/arc/projects/VVINGS/exgal_flows"
MODEL = f"{DATA}/rot_model/rad.mod.fits"
CO_PEAK = f"{DATA}/CO/m33_all_co21_2p5kms_8p0as_aligned_broad_tpeak.fits"
CACHE = Path("../outputs/highpass_check")
CACHE.mkdir(parents=True, exist_ok=True)

# label -> (mom1, mom0, config, decim).  decim is chosen so every epoch is
# sampled every ~12-14 pc (below each beam), keeping the three comparable.
# Ordered smallest to largest input grid, so the heavy 5.3" epoch runs last.
EPOCHS = {
    'C+D+GBT (21.4")': (
        f"{DATA}/HI/lglbs_CD/m33_C+D+tp_himidres_strict_mom1.fits",
        f"{DATA}/HI/lglbs_CD/m33_C+D+tp_himidres_strict_mom0.fits",
        "../config/m33.yaml", 1),
    '14B+17B (8.2")': (
        f"{DATA}/HI/17B/M33_14B_17B_HI_contsub_width_1kms.image.pbcor.GBT_feathered_K.mom1.fits",
        f"{DATA}/HI/17B/M33_14B_17B_HI_contsub_width_1kms.image.pbcor.GBT_feathered_K.mom0.fits",
        "../config/m33_old.yaml", 3),
    'A+B+C+D+GBT (5.3")': (
        f"{DATA}/HI/lglbs_ABCD/m33_A+B+C+D+GBT_hi21cm_strict_mom1.fits",
        f"{DATA}/HI/lglbs_ABCD/m33_A+B+C+D+GBT_hi21cm_strict_mom0.fits",
        "../config/m33.yaml", 4),
}

# name -> kwargs for FlowAnalysis.set_velocity_mode
MODES = {
    "rotation model": dict(mode="model"),
    "no subtraction": dict(mode="raw"),
    "high-pass 500 pc": dict(mode="highpass", highpass_fwhm_pc=500.0),
    "high-pass 1000 pc": dict(mode="highpass", highpass_fwhm_pc=1000.0),
}
# The package default (0.6 K) was tuned on the 7m-only peak map.  On this 8"
# map (max 3.4 K, median peak-noise ~0.03 K) it leaves ~no pixels above 2x, so
# the base cut is lowered: 0.1 K on the cos(i)-scaled map = 0.175 K on the sky
# (~6 sigma at the median noise).  Bins are then 2,3,4,7 x this.
CO_THRESHOLD_K = 0.1
EDGE_ERODE_PIX = 3   # same footprint erosion for every mode (see set_velocity_mode)

In [ ]:
def reference_medians(ana, decim):
    """Median divergence (km/s/kpc) of ALL pixels in the CO footprint.

    The control for the CO-bright medians: a large-scale offset shared by all
    gas shows up here too.  Returned for the whole aperture and per annulus.
    """
    an = ana.config.analysis
    sl = (slice(None, None, decim), slice(None, None, decim))
    div = ana.flow.divergence.value[sl] * 1000.0
    co = np.asarray(ana.co_deproj)[sl]
    rad_pc = ana.radius[sl] * ana.pixel_scale_pc
    ok = np.isfinite(div) & np.isfinite(co) & (rad_pc < an.radius_max_pc)
    edges = np.asarray(an.radial_edges_pc, float)
    per_annulus = np.array([
        np.median(div[ok & (rad_pc >= lo) & (rad_pc < hi)]) if np.any(ok & (rad_pc >= lo) & (rad_pc < hi)) else np.nan
        for lo, hi in zip(edges[:-1], edges[1:])
    ])
    return float(np.median(div[ok])), per_annulus, int(ok.sum())


def _mode_cache(tag, name):
    return CACHE / f"highpass_check_{tag}_{name.replace(' ', '_')}.pkl"


def _save_atomic(obj, path):
    """Pickle to a temp file, then rename: a kill mid-write leaves no corrupt checkpoint."""
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
    os.replace(tmp, path)


def release_memory():
    """Drop what keeps big arrays alive in a notebook, then collect garbage.

    After a failed cell IPython keeps the traceback (and so the frame holding
    ``ana``) in ``sys.last_*``; clear it so the arrays can be freed.
    """
    sys.last_type = sys.last_value = sys.last_traceback = None
    gc.collect()


def _compute_modes(label, tag, names):
    """Load one epoch and compute ``names``, checkpointing each mode as it finishes."""
    mom1, mom0, cfg, decim = EPOCHS[label]
    ana = None
    try:
        t0 = time.time()
        ana = FlowAnalysis.from_files(mom1_path=mom1, mom0_path=mom0, model_path=MODEL,
                                      co_paths={"hires": CO_PEAK}, config_path=cfg)
        ana.config.analysis.co_threshold_K = CO_THRESHOLD_K
        # only run() reads the sky-frame moment-0, and it is already deprojected
        ana.maps.mom0 = None
        pixel_scale_pc = ana.pixel_scale_pc
        print(f"{label}: {pixel_scale_pc:.2f} pc/pix, decim={decim}, loaded in {time.time()-t0:.0f}s")
        for name in names:
            t0 = time.time()
            ana.set_velocity_mode(edge_erode_pix=EDGE_ERODE_PIX, **MODES[name])
            # the statistics read only divergence and shear; free the raw gradients
            ana.flow.grad_x = ana.flow.grad_y = None
            ref_all, ref_annuli, n_ref = reference_medians(ana, decim)
            result = {
                "div": ana.result_divergence_of_co(decim=decim),
                "prof": ana.radial_divergence_profile(decim=decim, co_factor=2.0),
                "shear": ana.shear_sensitivity(decim=decim, n_boot=200),
                "co_frac": float(np.mean(np.asarray(ana.co_deproj)[np.isfinite(ana.co_deproj)] > 2 * CO_THRESHOLD_K)),
                "ref_all": ref_all, "ref_annuli": ref_annuli, "n_ref": n_ref,
            }
            _save_atomic({"pixel_scale_pc": pixel_scale_pc, "decim": decim, "result": result},
                         _mode_cache(tag, name))
            print(f"  {name:18s} done in {time.time()-t0:.0f}s")
    except BaseException as exc:
        # The traceback keeps every frame alive, including the pipeline method
        # that raised, whose ``self`` holds all the big arrays.  Clear their locals.
        traceback.clear_frames(exc.__traceback__)
        raise
    finally:
        del ana
        release_memory()


def run_epoch(label, force=False):
    """All velocity modes for one HI epoch.

    Every mode is checkpointed to ``CACHE`` as soon as it finishes
    (``highpass_check_<tag>_<mode>.pkl``), so a crash or out-of-memory kill
    loses only the mode in progress.  A rerun skips finished modes, and does
    not load the maps at all when every mode is done.  ``force=True``
    recomputes everything.  An epoch-level pickle from before per-mode
    checkpoints (``highpass_check_<tag>.pkl``) is used as it is.
    """
    tag = label.split()[0].replace("+", "")
    legacy = CACHE / f"highpass_check_{tag}.pkl"
    if legacy.exists() and not force:
        return pickle.load(open(legacy, "rb"))

    todo = [n for n in MODES if force or not _mode_cache(tag, n).exists()]
    if todo:
        _compute_modes(label, tag, todo)

    cached = {n: pickle.load(open(_mode_cache(tag, n), "rb")) for n in MODES}
    first = next(iter(cached.values()))
    return {"pixel_scale_pc": first["pixel_scale_pc"], "decim": first["decim"],
            "modes": {n: c["result"] for n, c in cached.items()}}


## Run (each mode is checkpointed to `outputs/highpass_check/`; pass `force=True` to recompute)


In [ ]:
import warnings
warnings.filterwarnings('ignore')
release_memory()
results = {label: run_epoch(label) for label in EPOCHS}

## Median divergence of CO-bright pixels

Rows are CO-brightness bins; `all pixels` is the control. Values are km/s/kpc
with the 68% wedge-bootstrap interval; negative means converging.

In [ ]:
from astropy.table import Table

rows = []
for label, res in results.items():
    for name, m in res["modes"].items():
        d = m["div"]
        row = {"epoch": label, "subtraction": name, "all pixels": f"{m['ref_all']:+.2f}"}
        for lab, med, lo, hi in zip(d.labels, d.medians, d.err_low, d.err_high):
            row[f"CO {lab}"] = f"{med:+.1f} [{lo:+.1f}, {hi:+.1f}]"
        rows.append(row)
div_table = Table(rows=rows)
div_table

In [ ]:
colors = dict(zip(MODES, ["#c1666b", "#8c8c8c", "#6baed6", "#1f4e79"]))
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (label, res) in zip(axes, results.items()):
    for k, (name, m) in enumerate(res["modes"].items()):
        d = m["div"]
        x = np.arange(len(d.labels)) + 0.18 * (k - 1.5)
        ax.errorbar(x, d.medians, yerr=[d.medians - d.err_low, d.err_high - d.medians],
                    fmt="o", ms=4, capsize=2, color=colors[name], label=name)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_xticks(np.arange(len(d.labels))); ax.set_xticklabels(d.labels)
    ax.set_title(label); ax.set_xlabel("CO peak / threshold")
axes[0].set_ylabel("median divergence of CO-bright [km/s/kpc]")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(CACHE / "divergence_by_subtraction.png", dpi=150)
plt.show()

## Radial profile of CO-bright divergence (CO > 2x threshold)

Points: CO-bright median per annulus with the tiled block-bootstrap interval.
Dotted lines: the all-pixel control in the same annulus and mode.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (label, res) in zip(axes, results.items()):
    for k, (name, m) in enumerate(res["modes"].items()):
        p = m["prof"]
        x = p.r_centers + 25 * (k - 1.5)
        ax.errorbar(x, p.medians, yerr=[p.medians - p.err_low, p.err_high - p.medians],
                    fmt="o-", ms=3, lw=0.8, capsize=2, color=colors[name], label=name)
        ax.plot(p.r_centers, m["ref_annuli"], ":", color=colors[name], lw=1.2)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(label); ax.set_xlabel("galactocentric radius [pc]")
axes[0].set_ylabel("median divergence [km/s/kpc]")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(CACHE / "radial_profile_by_subtraction.png", dpi=150)
plt.show()

rows = []
for label, res in results.items():
    for name, m in res["modes"].items():
        p = m["prof"]
        row = {"epoch": label, "subtraction": name, "turnover [pc]": f"{p.turnover_pc:.0f}"}
        for lo, hi, med in zip(p.r_edges[:-1], p.r_edges[1:], p.medians):
            row[f"{lo:.0f}-{hi:.0f}"] = f"{med:+.1f}"
        rows.append(row)
prof_table = Table(rows=rows)
prof_table

## Shear sensitivity

`rel_drop` = fractional fall in CO-bright fraction from the low-shear to the
high-shear half of the range, with the wedge-bootstrap interval.

In [ ]:
rows = []
for label, res in results.items():
    for name, m in res["modes"].items():
        s = m["shear"]
        rows.append({"epoch": label, "subtraction": name,
                     "lo_frac": round(s.lo_frac, 3), "hi_frac": round(s.hi_frac, 3),
                     "rel_drop": f"{s.rel_drop:.2f} [{s.rel_drop_lo:.2f}, {s.rel_drop_hi:.2f}]"})
shear_table = Table(rows=rows)
shear_table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, (label, res) in zip(axes, results.items()):
    for name, m in res["modes"].items():
        s = m["shear"]
        ax.plot(s.centers, s.fraction, "-", color=colors[name], label=name)
        ax.fill_between(s.centers, s.err_low, s.err_high, color=colors[name], alpha=0.2, lw=0)
    ax.set_title(label); ax.set_xlabel("|shear| [km/s/pc]")
axes[0].set_ylabel("CO-bright fraction")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(CACHE / "shear_by_subtraction.png", dpi=150)
plt.show()